# 01 · Op Fusion and Arithmetic Intensity

**Goal:** run the attention QKV projection (tokens × model weights) on a GPU and optimise it to better use the hardware, by fusing three matmuls into one. We measure how fusion changes *arithmetic intensity* (FLOPs per byte moved) and achieved FLOP/s on FP16 tensor cores as the number of tokens grows.

**Requirements:** NVIDIA GPU with tensor cores (compute capability ≥ 7.0) + CUDA + PyTorch. The notebook is GPU-agnostic: set your GPU's spec-sheet numbers in section 1 and everything else adapts.
On a machine without CUDA (e.g. a laptop) the code still runs for correctness, but performance numbers are meaningless and GPU-only cells are skipped.

## 0. Hardware

What GPU are we running on? We only need three facts: its **name**, its **compute capability** (the architecture generation, which decides what the tensor cores support), and its **DRAM** (VRAM) size.

In [1]:
import torch

GIGA = 1e9
TERA = 1e12

IS_CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if IS_CUDA_AVAILABLE else "cpu")

if IS_CUDA_AVAILABLE:
    properties = torch.cuda.get_device_properties(DEVICE)
    print(f"GPU                : {properties.name}")
    print(f"Compute capability : {properties.major}.{properties.minor} (SM{properties.major}{properties.minor})")
    print(f"DRAM               : {properties.total_memory / GIGA:.1f} GB")
else:
    print("No CUDA GPU found: running on CPU for correctness only.")

No CUDA GPU found: running on CPU for correctness only.


## 1. Spec Sheet

PyTorch can't tell us peak throughput, so we take it from the GPU's datasheet. Defaults below are the **NVIDIA T4** (Turing, SM75).

| T4 datasheet | Value |
|---|---|
| Peak FP16 tensor core | 65 TFLOP/s |
| Peak INT8 tensor core | 130 TOP/s |
| Peak INT4 tensor core | 260 TOP/s |
| Peak FP32 (no tensor cores) | 8.1 TFLOP/s |
| Memory bandwidth | 320 GB/s (GDDR6) |

SM75 tensor cores natively support **FP16, INT8 and INT4**. We use **FP16**: it's the floating-point format the tensor cores run natively, so our matmuls can reach the 65 TFLOP/s peak.

✏️ On a different GPU, replace these values with its datasheet numbers and pick a precision its tensor cores support.

In [3]:
PEAK_TENSOR_CORE_FLOPS_PER_SECOND = 65 * TERA
PEAK_MEMORY_BANDWIDTH_BYTES_PER_SECOND = 320 * GIGA
DTYPE = torch.float16

## 2. The Operation

A model represents each token (a word or word-piece) as a list of `d` numbers, called its **hidden state**. `d` is the **hidden size**; we use `d = 768` (GPT-2 small).

Attention needs three versions of every token, a **query**, a **key** and a **value**, each obtained by multiplying the token's hidden state by a fixed weight matrix of the model. Stacking `N` tokens as rows of a matrix `X`:

```
X          : [N, d]      N tokens, d numbers each
Wq, Wk, Wv : [d, d]      model weights
Q = X @ Wq,   K = X @ Wk,   V = X @ Wv      each [N, d]
```

In PyTorch `@` is **matrix multiplication** (`torch.matmul`), not element-wise `*`.

### Where the cost comes from

DRAM (the GPU's memory) and the compute units are separate, connected by a memory bus. On a T4 that bus moves 320 GB/s. Every matrix multiplication must bring its inputs across the bus to compute, and send its result back to DRAM. The math is fast; the travel is often what limits speed. Fusing is about doing the same math with less travel.

In [4]:
HIDDEN_SIZE = 768
TOKEN_COUNT = 128
RANDOM_SEED = 0

torch.manual_seed(RANDOM_SEED)
X = torch.randn(TOKEN_COUNT, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)
W_q = torch.randn(HIDDEN_SIZE, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE) / HIDDEN_SIZE ** 0.5
W_k = torch.randn(HIDDEN_SIZE, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE) / HIDDEN_SIZE ** 0.5
W_v = torch.randn(HIDDEN_SIZE, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE) / HIDDEN_SIZE ** 0.5

print(f"X: {tuple(X.shape)} | Wq, Wk, Wv: {tuple(W_q.shape)} each | dtype: {DTYPE}")

X: (128, 768) | Wq, Wk, Wv: (768, 768) each | dtype: torch.float16


Weights are divided by `√d` (standard initialisation) so the FP16 outputs stay in a sensible range.

## 3. Unfused: Q, K, V Computed Separately

Three separate multiplications. Each one is a round trip over the bus: inputs travel DRAM → compute, the result travels compute → DRAM.

| Multiplication | Travels to compute | Travels back to DRAM |
|---|---|---|
| `X @ Wq` | `X`, `Wq` | `Q` |
| `X @ Wk` | `X`, `Wk` | `K` |
| `X @ Wv` | `X`, `Wv` | `V` |

3 round trips = **6 travels**, and `X` crosses the bus **3 times** even though it's the same data every time.

In [4]:
Q_unfused = X @ W_q
K_unfused = X @ W_k
V_unfused = X @ W_v

print(f"Q, K, V: {tuple(Q_unfused.shape)} each")

Q, K, V: (128, 768) each


## 4. Fused: One Multiplication

Place the three weights side by side into one `[d, 3d]` matrix. A single multiplication then produces `Q`, `K` and `V` together as one `[N, 3d]` result: its first `d` columns are `Q`, the next `d` are `K`, the last `d` are `V`.

```
W_qkv = [Wq | Wk | Wv]        [d, 3d]   built once, when the model is loaded
QKV   = X @ W_qkv             [N, 3d]
```

| Multiplication | Travels to compute | Travels back to DRAM |
|---|---|---|
| `X @ W_qkv` | `X`, `W_qkv` | `QKV` |

1 round trip = **2 travels**. `X` crosses the bus **once**.

Note what *didn't* shrink: `W_qkv` is the same bytes as `Wq + Wk + Wv`, and `QKV` the same bytes as `Q + K + V`. The saving is exactly the two extra trips of `X`. How much that matters depends on how big `X` is relative to the weights; the next section measures it.

In [5]:
W_qkv = torch.cat([W_q, W_k, W_v], dim=1)

QKV = X @ W_qkv
Q_fused, K_fused, V_fused = QKV.split(HIDDEN_SIZE, dim=1)

print(f"W_qkv: {tuple(W_qkv.shape)} | QKV: {tuple(QKV.shape)}")

W_qkv: (768, 2304) | QKV: (128, 2304)


Check that fusing didn't change the result. Tiny FP16 differences are allowed since the multiplications may add terms in a different order.

In [6]:
FP16_TOLERANCE = 1e-2

torch.testing.assert_close(Q_fused, Q_unfused, atol=FP16_TOLERANCE, rtol=FP16_TOLERANCE)
torch.testing.assert_close(K_fused, K_unfused, atol=FP16_TOLERANCE, rtol=FP16_TOLERANCE)
torch.testing.assert_close(V_fused, V_unfused, atol=FP16_TOLERANCE, rtol=FP16_TOLERANCE)
print("Fused and unfused Q, K, V match ✓")

Fused and unfused Q, K, V match ✓
